In [1]:
# ============================================================
# TCGPLAYER DYNAMIC PRICE MONITOR
# ============================================================

import asyncio
import json
import os
import re
import time

import requests
from playwright.async_api import async_playwright


# ============================================================
# CONFIGURATION
# ============================================================

TRACKED_PRODUCTS = [
    {
        "product_id": 707591,
        "name": "Lacus EX Resource"
    },
    {
        "product_id": 707586,
        "name": "Sayla Mass EX Resource"
    },
    {
        "product_id": 707585,
        "name": "Miorine EX Resource"
    }
]


# Discord webhook should be stored as an environment variable.
DISCORD_WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL")


# Check every 5 minutes
CHECK_INTERVAL_SECONDS = 300


# Alert when total price is below 70% of TCG Market Price
DYNAMIC_PERCENTAGE = 0.70


# Maximum number of listings to evaluate
MAX_LISTINGS_TO_CHECK = 5


# Small pause between products
PRODUCT_DELAY_SECONDS = 2


# ============================================================
# VALIDATE CONFIGURATION
# ============================================================

def validate_configuration():
    """
    Validate important configuration settings before monitoring starts.
    """

    if not DISCORD_WEBHOOK_URL:
        print(
            "⚠️ DISCORD_WEBHOOK_URL is not configured.\n"
            "Price monitoring will still work, but Discord alerts "
            "cannot be sent."
        )

    if not TRACKED_PRODUCTS:
        raise ValueError("TRACKED_PRODUCTS cannot be empty.")

    if not 0 < DYNAMIC_PERCENTAGE < 1:
        raise ValueError(
            "DYNAMIC_PERCENTAGE must be between 0 and 1."
        )


# ============================================================
# BUILD TCGPLAYER URL
# ============================================================

def build_product_url(product_id):
    """
    Generate the TCGplayer URL for a product.
    """

    return (
        f"https://www.tcgplayer.com/product/{product_id}"
        f"?Language=English"
    )


# ============================================================
# DISCORD ALERT
# ============================================================

def send_discord_alert_with_file(
    product_id,
    name,
    market_price,
    target_price,
    lowest_price,
    shipping,
    total,
    condition,
    seller,
    screenshot_path
):
    """
    Send a Discord webhook alert when a listing falls below
    the calculated target price.
    """

    if not DISCORD_WEBHOOK_URL:
        print(
            "⚠️ Discord webhook is not configured. "
            "Skipping Discord notification.",
            flush=True
        )
        return

    product_url = build_product_url(product_id)

    embed_data = {
        "title": "🚨 Dynamic Price Drop Alert! (Below 70% of Market)",
        "url": product_url,
        "color": 15158332,

        "fields": [
            {
                "name": "Total Cost",
                "value": f"**${total:.2f}**",
                "inline": True
            },
            {
                "name": "Dynamic Target (70%)",
                "value": f"${target_price:.2f}",
                "inline": True
            },
            {
                "name": "TCG Market Price",
                "value": f"${market_price:.2f}",
                "inline": True
            },
            {
                "name": "Base Price",
                "value": f"${lowest_price:.2f}",
                "inline": True
            },
            {
                "name": "Shipping",
                "value": f"${shipping:.2f}",
                "inline": True
            },
            {
                "name": "Condition",
                "value": condition.strip(),
                "inline": False
            },
            {
                "name": "Seller",
                "value": seller.strip(),
                "inline": False
            }
        ],

        "footer": {
            "text": (
                f"Product ID: {product_id} | "
                f"TCGplayer Dynamic Price Monitor"
            )
        }
    }

    payload_json = {
        "content": (
            f"🚨 **TCGplayer Target Price Triggered "
            f"for {name}!** 🚨"
        ),
        "embeds": [embed_data]
    }

    try:

        # ----------------------------------------------------
        # SEND WITHOUT IMAGE
        # ----------------------------------------------------

        if not os.path.exists(screenshot_path):

            print(
                "⚠️ Screenshot unavailable. "
                "Sending text-only Discord alert.",
                flush=True
            )

            response = requests.post(
                DISCORD_WEBHOOK_URL,
                json=payload_json,
                timeout=20
            )

        # ----------------------------------------------------
        # SEND WITH IMAGE
        # ----------------------------------------------------

        else:

            embed_data["image"] = {
                "url": "attachment://card_art.png"
            }

            with open(screenshot_path, "rb") as image_file:

                files = {
                    "file": (
                        "card_art.png",
                        image_file,
                        "image/png"
                    )
                }

                data = {
                    "payload_json": json.dumps(payload_json)
                }

                response = requests.post(
                    DISCORD_WEBHOOK_URL,
                    data=data,
                    files=files,
                    timeout=20
                )

        # ----------------------------------------------------
        # RESPONSE CHECK
        # ----------------------------------------------------

        if response.status_code in (200, 204):

            print(
                f"🚀 Discord alert sent successfully for {name}.",
                flush=True
            )

        else:

            print(
                f"❌ Discord returned status "
                f"{response.status_code}.",
                flush=True
            )

            print(
                response.text,
                flush=True
            )

    except requests.RequestException as error:

        print(
            f"❌ Discord request failed: {error}",
            flush=True
        )


# ============================================================
# EXTRACT MARKET PRICE
# ============================================================

async def extract_market_price(page):
    """
    Attempt multiple strategies for extracting TCGplayer's
    displayed Market Price.
    """

    body_text = await page.locator("body").inner_text()

    clean_body_text = (
        body_text
        .replace("\u00a0", " ")
        .replace("\u202f", " ")
    )

    clean_body_text = re.sub(
        r"[ \t]+",
        " ",
        clean_body_text
    )

    # --------------------------------------------------------
    # METHOD 1:
    # Market Price followed by price
    # --------------------------------------------------------

    match = re.search(
        r"Market\s+Price"
        r"\s*[:\-]?\s*"
        r"\$\s*"
        r"([0-9,]+\.[0-9]{2})",
        clean_body_text,
        re.IGNORECASE
    )

    if match:

        market_price = float(
            match.group(1).replace(",", "")
        )

        print(
            f"✅ Market Price found using Method 1: "
            f"${market_price:.2f}",
            flush=True
        )

        return market_price

    # --------------------------------------------------------
    # METHOD 2:
    # Price followed by Market Price
    # --------------------------------------------------------

    match = re.search(
        r"\$\s*"
        r"([0-9,]+\.[0-9]{2})"
        r"\s*"
        r"Market\s+Price",
        clean_body_text,
        re.IGNORECASE
    )

    if match:

        market_price = float(
            match.group(1).replace(",", "")
        )

        print(
            f"✅ Market Price found using Method 2: "
            f"${market_price:.2f}",
            flush=True
        )

        return market_price

    # --------------------------------------------------------
    # METHOD 3:
    # Search parent containers near "Market Price"
    # --------------------------------------------------------

    market_locator = page.get_by_text(
        "Market Price",
        exact=False
    ).first

    if await market_locator.count() > 0:

        current = market_locator

        for level in range(6):

            try:

                container_text = await current.inner_text()

                prices = re.findall(
                    r"\$\s*([0-9,]+\.[0-9]{2})",
                    container_text
                )

                if prices:

                    market_price = float(
                        prices[0].replace(",", "")
                    )

                    print(
                        f"✅ Market Price found near label: "
                        f"${market_price:.2f}",
                        flush=True
                    )

                    return market_price

                current = current.locator("..")

            except Exception:
                break

    # --------------------------------------------------------
    # METHOD 4:
    # Search text immediately around Market Price
    # --------------------------------------------------------

    market_position = clean_body_text.lower().find(
        "market price"
    )

    if market_position >= 0:

        nearby_text = clean_body_text[
            market_position:
            market_position + 250
        ]

        prices = re.findall(
            r"\$\s*([0-9,]+\.[0-9]{2})",
            nearby_text
        )

        if prices:

            market_price = float(
                prices[0].replace(",", "")
            )

            print(
                f"✅ Market Price found from nearby text: "
                f"${market_price:.2f}",
                flush=True
            )

            return market_price

    return 0.0


# ============================================================
# MARKET PRICE DEBUG
# ============================================================

async def print_market_price_debug(page):
    """
    Print text surrounding 'Market Price' if extraction fails.
    Useful when TCGplayer changes its website structure.
    """

    try:

        body_text = await page.locator("body").inner_text()

        position = body_text.lower().find(
            "market price"
        )

        if position >= 0:

            start = max(
                0,
                position - 400
            )

            end = min(
                len(body_text),
                position + 600
            )

            print(
                "\n========== MARKET PRICE DEBUG ==========",
                flush=True
            )

            print(
                body_text[start:end],
                flush=True
            )

            print(
                "========================================\n",
                flush=True
            )

        else:

            print(
                "⚠️ 'Market Price' was not found "
                "in the rendered page text.",
                flush=True
            )

    except Exception as error:

        print(
            f"⚠️ Market Price debug failed: {error}",
            flush=True
        )


# ============================================================
# CAPTURE PRODUCT IMAGE
# ============================================================

async def capture_product_image(
    page,
    screenshot_path
):
    """
    Attempt to capture the product image for Discord alerts.
    """

    image_selectors = [
        "div.product-details__image",
        ".product-details__image img",
        ".lazy-image img",
        ".lazy-image",
        "img[alt*='Product Image']",
        "img[alt*='product']"
    ]

    for selector in image_selectors:

        try:

            candidate = page.locator(
                selector
            ).first

            if (
                await candidate.count() > 0
                and await candidate.is_visible()
            ):

                await candidate.screenshot(
                    path=screenshot_path
                )

                return

        except Exception:
            continue

    # Fallback
    try:

        print(
            "⚠️ Product image not found. "
            "Using viewport screenshot.",
            flush=True
        )

        await page.screenshot(
            path=screenshot_path,
            full_page=False
        )

    except Exception as error:

        print(
            f"⚠️ Screenshot failed: {error}",
            flush=True
        )


# ============================================================
# PARSE LISTING
# ============================================================

async def parse_listing(item):
    """
    Parse price, shipping, condition, and seller
    from one TCGplayer listing.
    """

    card_text = await item.inner_text()

    # --------------------------------------------------------
    # BASE PRICE
    # --------------------------------------------------------

    price_match = re.search(
        r"\$\s*([0-9,]+\.[0-9]{2})",
        card_text
    )

    if not price_match:
        return None

    price = float(
        price_match
        .group(1)
        .replace(",", "")
    )

    if price <= 0:
        return None

    # --------------------------------------------------------
    # SHIPPING
    # --------------------------------------------------------

    shipping = 0.0

    shipping_match = re.search(
        r"(?:"
        r"\+\s*\$\s*([0-9,]+\.[0-9]{2})"
        r"|"
        r"free\s+shipping"
        r")",
        card_text,
        re.IGNORECASE
    )

    if shipping_match:

        if "free" in shipping_match.group(0).lower():

            shipping = 0.0

        elif shipping_match.group(1):

            shipping = float(
                shipping_match
                .group(1)
                .replace(",", "")
            )

    elif (
        "free" in card_text.lower()
        and "shipping" in card_text.lower()
    ):

        shipping = 0.0

    else:

        all_prices = re.findall(
            r"\$\s*([0-9,]+\.[0-9]{2})",
            card_text
        )

        if len(all_prices) >= 2:

            shipping = float(
                all_prices[1].replace(",", "")
            )

    # --------------------------------------------------------
    # CONDITION
    # --------------------------------------------------------

    condition_element = await item.query_selector(
        ".listing-item__condition, "
        "[class*='condition']"
    )

    if condition_element:

        condition = (
            await condition_element.inner_text()
        ).strip()

    else:

        condition = "Not Available"

    # --------------------------------------------------------
    # SELLER
    # --------------------------------------------------------

    seller_element = await item.query_selector(
        ".seller-info__name, "
        "[class*='seller']"
    )

    if seller_element:

        seller = (
            await seller_element.inner_text()
        ).strip()

    else:

        seller = "Marketplace Seller"

    total = price + shipping

    return {
        "price": price,
        "shipping": shipping,
        "total": total,
        "condition": condition,
        "seller": seller
    }


# ============================================================
# SCRAPE ONE PRODUCT
# ============================================================

async def scrape_single_product(
    context,
    product_info
):
    """
    Monitor a single TCGplayer product.
    """

    product_id = product_info["product_id"]
    name = product_info["name"]

    url = build_product_url(product_id)

    screenshot_path = (
        f"card_art_{product_id}.png"
    )

    print(
        f"\n🔍 Processing: {name} "
        f"(Product ID: {product_id})",
        flush=True
    )

    page = await context.new_page()

    try:

        # ----------------------------------------------------
        # OPEN PRODUCT PAGE
        # ----------------------------------------------------

        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        await page.wait_for_timeout(
            3000
        )

        # ----------------------------------------------------
        # WAIT FOR LISTINGS
        # ----------------------------------------------------

        print(
            "⏳ Waiting for marketplace listings...",
            flush=True
        )

        await page.wait_for_selector(
            ".listing-item",
            timeout=25000
        )

        # ----------------------------------------------------
        # WAIT FOR MARKET PRICE
        # ----------------------------------------------------

        print(
            "⏳ Waiting for Market Price summary...",
            flush=True
        )

        try:

            await page.get_by_text(
                "Market Price",
                exact=False
            ).first.wait_for(
                state="visible",
                timeout=15000
            )

        except Exception:

            print(
                "⚠️ Market Price label did not become "
                "visible within timeout.",
                flush=True
            )

        # ----------------------------------------------------
        # EXTRACT MARKET PRICE
        # ----------------------------------------------------

        print(
            "🔎 Extracting Market Price...",
            flush=True
        )

        market_price = await extract_market_price(
            page
        )

        if market_price <= 0:

            print(
                f"❌ Could not determine Market Price "
                f"for {name}.",
                flush=True
            )

            await print_market_price_debug(
                page
            )

            return

        # ----------------------------------------------------
        # TARGET PRICE
        # ----------------------------------------------------

        calculated_target_price = (
            market_price
            * DYNAMIC_PERCENTAGE
        )

        print(
            f"📊 TCG Market Price: "
            f"${market_price:.2f}",
            flush=True
        )

        print(
            f"🎯 Dynamic Target "
            f"({DYNAMIC_PERCENTAGE * 100:.0f}%): "
            f"${calculated_target_price:.2f}",
            flush=True
        )

        # ----------------------------------------------------
        # PRODUCT IMAGE
        # ----------------------------------------------------

        await capture_product_image(
            page,
            screenshot_path
        )

        # ----------------------------------------------------
        # GET LISTINGS
        # ----------------------------------------------------

        listings = await page.query_selector_all(
            ".listing-item"
        )

        if not listings:

            print(
                f"⚠️ No listings found for {name}.",
                flush=True
            )

            return

        print(
            f"📦 Found {len(listings)} listings.",
            flush=True
        )

        lowest_listing = None

        # ----------------------------------------------------
        # PROCESS LISTINGS
        # ----------------------------------------------------

        for index, item in enumerate(
            listings[:MAX_LISTINGS_TO_CHECK],
            start=1
        ):

            try:

                listing = await parse_listing(
                    item
                )

                if listing is None:
                    continue

                print(
                    f"   Listing #{index}: "
                    f"${listing['price']:.2f} "
                    f"+ ${listing['shipping']:.2f} shipping "
                    f"= ${listing['total']:.2f}",
                    flush=True
                )

                if (
                    lowest_listing is None
                    or listing["total"]
                    < lowest_listing["total"]
                ):

                    lowest_listing = listing

            except Exception as error:

                print(
                    f"⚠️ Failed parsing listing "
                    f"#{index}: {error}",
                    flush=True
                )

        # ----------------------------------------------------
        # VALIDATE LISTINGS
        # ----------------------------------------------------

        if lowest_listing is None:

            print(
                f"⚠️ No valid listings could be parsed "
                f"for {name}.",
                flush=True
            )

            return

        # ----------------------------------------------------
        # DISPLAY LOWEST PRICE
        # ----------------------------------------------------

        print(
            f"💵 Lowest Total Cost: "
            f"${lowest_listing['total']:.2f}",
            flush=True
        )

        print(
            f"🎯 Alert Target: "
            f"${calculated_target_price:.2f}",
            flush=True
        )

        # ----------------------------------------------------
        # TARGET CHECK
        # ----------------------------------------------------

        if (
            lowest_listing["total"]
            < calculated_target_price
        ):

            print(
                f"🚨 TARGET HIT FOR {name}!",
                flush=True
            )

            print(
                f"💵 Lowest: "
                f"${lowest_listing['total']:.2f}",
                flush=True
            )

            print(
                f"🎯 Target: "
                f"${calculated_target_price:.2f}",
                flush=True
            )

            print(
                "📨 Sending Discord alert...",
                flush=True
            )

            send_discord_alert_with_file(
                product_id=product_id,
                name=name,
                market_price=market_price,
                target_price=calculated_target_price,
                lowest_price=lowest_listing["price"],
                shipping=lowest_listing["shipping"],
                total=lowest_listing["total"],
                condition=lowest_listing["condition"],
                seller=lowest_listing["seller"],
                screenshot_path=screenshot_path
            )

        else:

            print(
                f"😴 No alert. "
                f"${lowest_listing['total']:.2f} "
                f"is above target "
                f"${calculated_target_price:.2f}.",
                flush=True
            )

    except Exception as error:

        print(
            f"❌ Error monitoring {name}: "
            f"{error}",
            flush=True
        )

    finally:

        # ----------------------------------------------------
        # DELETE TEMPORARY SCREENSHOT
        # ----------------------------------------------------

        if os.path.exists(
            screenshot_path
        ):

            try:

                os.remove(
                    screenshot_path
                )

            except OSError:
                pass

        # ----------------------------------------------------
        # CLOSE PRODUCT PAGE
        # ----------------------------------------------------

        try:

            if not page.is_closed():

                await page.close()

        except Exception:
            pass


# ============================================================
# CREATE BROWSER CONTEXT
# ============================================================

async def create_browser_context(
    playwright
):
    """
    Create Chromium browser and browsing context.
    """

    browser = await playwright.chromium.launch(
        headless=True
    )

    context = await browser.new_context(

        viewport={
            "width": 1440,
            "height": 1200
        },

        user_agent=(
            "Mozilla/5.0 "
            "(Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/140.0.0.0 "
            "Safari/537.36"
        ),

        locale="en-US"
    )

    return browser, context


# ============================================================
# RUN ONE MONITORING CYCLE
# ============================================================

async def run_once():
    """
    Check all configured products one time.

    Recommended for testing and demonstrating the notebook
    on GitHub.
    """

    validate_configuration()

    print(
        "\n🚀 Starting TCGplayer Price Monitor",
        flush=True
    )

    print(
        f"📦 Tracking "
        f"{len(TRACKED_PRODUCTS)} products",
        flush=True
    )

    print(
        f"🎯 Alert threshold: "
        f"{DYNAMIC_PERCENTAGE * 100:.0f}% "
        f"of Market Price",
        flush=True
    )

    async with async_playwright() as p:

        browser, context = (
            await create_browser_context(p)
        )

        try:

            for product in TRACKED_PRODUCTS:

                await scrape_single_product(
                    context,
                    product
                )

                await asyncio.sleep(
                    PRODUCT_DELAY_SECONDS
                )

        finally:

            await context.close()
            await browser.close()

            print(
                "\n✅ Monitoring cycle complete.",
                flush=True
            )


# ============================================================
# CONTINUOUS MONITORING
# ============================================================

async def main_loop():
    """
    Continuously monitor all configured products.
    """

    validate_configuration()

    print(
        "\n🚀 Starting TCGplayer Dynamic Price Monitor",
        flush=True
    )

    print(
        f"📦 Tracking "
        f"{len(TRACKED_PRODUCTS)} products",
        flush=True
    )

    print(
        f"🎯 Alert threshold: "
        f"{DYNAMIC_PERCENTAGE * 100:.0f}% "
        f"of Market Price",
        flush=True
    )

    print(
        f"⏱️ Check interval: "
        f"{CHECK_INTERVAL_SECONDS} seconds",
        flush=True
    )

    async with async_playwright() as p:

        browser, context = (
            await create_browser_context(p)
        )

        cycle_number = 1

        try:

            while True:

                print(
                    "\n" + "=" * 70,
                    flush=True
                )

                print(
                    f"🔄 MONITORING CYCLE "
                    f"#{cycle_number}",
                    flush=True
                )

                print(
                    "=" * 70,
                    flush=True
                )

                cycle_start = time.time()

                for product in TRACKED_PRODUCTS:

                    try:

                        await scrape_single_product(
                            context,
                            product
                        )

                    except Exception as error:

                        print(
                            f"❌ Unexpected error for "
                            f"{product['name']}: "
                            f"{error}",
                            flush=True
                        )

                    await asyncio.sleep(
                        PRODUCT_DELAY_SECONDS
                    )

                cycle_duration = (
                    time.time()
                    - cycle_start
                )

                print(
                    "\n✅ Monitoring cycle complete.",
                    flush=True
                )

                print(
                    f"⏱️ Cycle duration: "
                    f"{cycle_duration:.1f} seconds.",
                    flush=True
                )

                print(
                    f"💤 Waiting "
                    f"{CHECK_INTERVAL_SECONDS} seconds "
                    f"before next check...",
                    flush=True
                )

                cycle_number += 1

                await asyncio.sleep(
                    CHECK_INTERVAL_SECONDS
                )

        except asyncio.CancelledError:

            print(
                "\n🛑 Monitoring task cancelled.",
                flush=True
            )

        except KeyboardInterrupt:

            print(
                "\n🛑 Monitoring stopped.",
                flush=True
            )

        finally:

            print(
                "🧹 Closing browser...",
                flush=True
            )

            await context.close()
            await browser.close()

            print(
                "✅ Browser closed.",
                flush=True
            )

In [3]:
# ============================================================
# For the GitHub demonstration only
# ============================================================

await run_once()

⚠️ DISCORD_WEBHOOK_URL is not configured.
Price monitoring will still work, but Discord alerts cannot be sent.

🚀 Starting TCGplayer Price Monitor
📦 Tracking 3 products
🎯 Alert threshold: 70% of Market Price

🔍 Processing: Lacus EX Resource (Product ID: 707591)
⏳ Waiting for marketplace listings...
⏳ Waiting for Market Price summary...
🔎 Extracting Market Price...
✅ Market Price found using Method 1: $854.61
📊 TCG Market Price: $854.61
🎯 Dynamic Target (70%): $598.23
⚠️ Product image not found. Using viewport screenshot.
📦 Found 10 listings.
   Listing #1: $784.00 + $0.99 shipping = $784.99
   Listing #2: $785.95 + $0.00 shipping = $785.95
   Listing #3: $789.99 + $0.00 shipping = $789.99
   Listing #4: $789.92 + $1.99 shipping = $791.91
   Listing #5: $799.99 + $0.00 shipping = $799.99
💵 Lowest Total Cost: $784.99
🎯 Alert Target: $598.23
😴 No alert. $784.99 is above target $598.23.

🔍 Processing: Sayla Mass EX Resource (Product ID: 707586)
⏳ Waiting for marketplace listings...
⏳ Waiti

In [ ]:
# ============================================================
# For your actual continuous monitor:
# ============================================================

await main_loop()